# NumPy Indexing & Slicing

> 📘 **Python Mastery** · Module 10 — NumPy · Lesson 2/7

Data is useless until you can select the exact pieces you need. This lesson covers single elements, slices, rows and columns, boolean masks and fancy indexing — plus the view-vs-copy trap that surprises almost everyone exactly once.

## 🎯 Learning Objectives

- Access single elements of 1-D and 2-D arrays, including negative positions
- Slice arrays with `[start:stop:step]` and extract whole rows and columns
- Explain why slices are views and protect yourself with `.copy()`
- Filter arrays with boolean masks combined using `&`, `|` and `~`
- Select arbitrary elements with fancy indexing and mix it with slicing
- Update many elements at once through masked assignment

## 1. Indexing 1-D Arrays

Square brackets grab single elements. Positions start at 0, and negative positions count from the end — `[-1]` is always the last element.

**Syntax:**
```python
arr[0]     # first element
arr[-1]    # last element
arr[i]     # the (i+1)-th element; valid while -len(arr) <= i < len(arr)
```

In [ ]:
import numpy as np

scores = np.array([78, 92, 61, 88, 95, 73])   # six quiz scores

print("first      :", scores[0])
print("second     :", scores[1])
print("last       :", scores[-1])
print("second last:", scores[-2])
print("positions run from 0 to", len(scores) - 1, "- shape is", scores.shape)

## 2. Slicing: start:stop:step

The slice syntax works just like list slicing: start included, stop excluded, optional step. A slice answers "give me a stretch of the array".

**Syntax:**
```python
arr[start:stop]     # positions start .. stop-1
arr[start:]         # from start to the end
arr[:stop]          # from the beginning up to stop-1
arr[start:stop:2]   # every other element
arr[::-1]           # reversed order
```

In [ ]:
import numpy as np

scores = np.array([78, 92, 61, 88, 95, 73])

print("scores[1:4] :", scores[1:4])   # positions 1, 2, 3 - stop EXCLUDED
print("scores[:3]  :", scores[:3])    # first three
print("scores[-2:] :", scores[-2:])   # last two
print("scores[::2] :", scores[::2])   # every other score
print("scores[::-1]:", scores[::-1])  # reversed

## 3. ⚠️ Slices Are Views, Not Copies!

Here is the trap. In plain Python, slicing a list makes an independent copy. In NumPy, a slice is a **view**: a lightweight window onto the SAME memory as the original. Modify the view, and the original changes too.

**Syntax:**
```python
b = arr[1:4]        # VIEW - shares data with arr
c = arr[1:4].copy() # independent COPY - safe to modify
```

In [ ]:
import numpy as np

scores = np.array([78, 92, 61, 88, 95])

top = scores[2:5]      # looks like a fresh little list...
top[0] = 999           # ...but we just edited the ORIGINAL

print("slice :", top)
print("source:", scores)   # the 999 leaked into scores!

In [ ]:
import numpy as np

scores = np.array([78, 92, 61, 88, 95])

safe = scores[2:5].copy()   # .copy() detaches the slice from its source
safe[0] = 999

print("copy  :", safe)
print("source:", scores)     # untouched this time

> 🔍 **Under the Hood:** Slicing never touches the data buffer. NumPy returns a new small header object that points into the SAME memory as the original, with its own shape and *strides* describing which region to look at. Copying a million-element slice would cost megabytes and milliseconds; creating a view costs about a hundred bytes and nanoseconds. Prove it yourself with `np.shares_memory(scores, top)` → `True`. This design is why pandas column selections feel instant — they are usually views too.

## 4. Indexing 2-D Arrays: arr[row, col]

For matrices, pass both positions inside ONE pair of brackets: `arr[row, col]`. The double-bracket style `arr[row][col]` also works but does two separate lookups — avoid it.

**Syntax:**
```python
m[r, c]      # preferred: one lookup with a tuple index
m[r][c]      # works, but slower: first grabs row r, then indexes it
m[-1, 0]     # negatives work on either axis
```

In [ ]:
import numpy as np

grades = np.array([[78, 85, 90],     # student 0: three subjects
                   [62, 71, 88],     # student 1
                   [90, 93, 79]])    # student 2

print("grades[1, 2] :", grades[1, 2])   # row 1, col 2 -> 88 (preferred form)
print("grades[1][2] :", grades[1][2])   # same value, but two lookups
print("grades[-1, 0] :", grades[-1, 0]) # last row, first column

## 5. Whole Rows and Columns with Slices

Leaving an axis as `:` means "everything along that axis". `grades[1, :]` is the whole second row; `grades[:, 0]` is the whole first column. Two ranges give you a rectangular sub-block.

**Syntax:**
```python
m[r, :]         # entire row r
m[:, c]         # entire column c
m[r0:r1, c0:c1] # rectangular sub-block
```

In [ ]:
import numpy as np

grades = np.array([[78, 85, 90],
                   [62, 71, 88],
                   [90, 93, 79]])

print("all of row 1   :", grades[1, :])
print("all of column 0:", grades[:, 0], "(first subject, all students)")
print()
print("first two rows, first two columns (a 2x2 sub-block):")
print(grades[0:2, 0:2])

## 6. Boolean Mask Filtering

A **boolean mask** is an array of True/False values with the same shape as the data. Indexing with it keeps only the True positions. This is THE pattern for filtering datasets — you will use it daily in pandas too.

**Syntax:**
```python
mask = arr > threshold                  # build a boolean array
filtered = arr[mask]                    # keep True positions
filtered = arr[arr > threshold]         # same thing, one line
both = arr[(arr > low) & (arr < high)]  # combine with & | ~ + parentheses
```

Important: use `&`, `|`, `~` — NOT the keywords `and`, `or`, `not`, and always wrap each comparison in parentheses.

In [ ]:
import numpy as np

temps = np.array([31.5, 32.0, 33.8, 30.2, 29.9, 34.1])

hot_days = temps > 32            # one verdict per element
print("mask     :", hot_days)
print("hot temps:", temps[hot_days])

# The idiomatic one-line form: read it as "temps where temps > 32"
print("direct   :", temps[temps > 32])

In [ ]:
import numpy as np

scores = np.array([45, 82, 67, 91, 58, 73, 39, 88])

mid_range   = scores[(scores >= 50) & (scores < 80)]
high_or_low = scores[(scores >= 80) | (scores < 40)]
not_failing = scores[~(scores < 40)]

print("50 to 79    :", mid_range)
print(">=80 or <40 :", high_or_low)
print("passed      :", not_failing)
print("passed count:", np.count_nonzero(scores >= 50))

## 7. Fancy Indexing with Integer Arrays

Pass a LIST or array of positions and NumPy returns those elements — in your order, duplicates allowed. Perfect for "give me items 3, 0 and 3" or reordering records.

On 2-D arrays, pairing two index arrays selects coordinates: `m[[r0, r1], [c0, c1]]` returns `m[r0, c0]` and `m[r1, c1]`.

**Syntax:**
```python
arr[[0, 3, 1]]          # elements at those positions, in that order
m[[0, 2], [1, 0]]       # pairs: (0,1) then (2,0)
```

In [ ]:
import numpy as np

menu_prices = np.array([150, 220, 90, 310, 175])

print("picked prices:", menu_prices[[3, 0, 3]])   # repeats allowed
print("reordered    :", menu_prices[[4, 2, 0, 1, 3]])

grid = np.array([[10, 20, 30],
                 [40, 50, 60],
                 [70, 80, 90]])
print("corner picks :", grid[[0, 2], [0, 2]])    # (0,0)=10 and (2,2)=90

## 8. Combining Fancy Indexing with Slices

Fancy indices mix freely with slices across axes: slice one axis, hand-pick on the other. Read `grades[[0, 2], 1:3]` as "students 0 and 2, subjects 1 through 2".

**Syntax:**
```python
m[[0, 2], 1:3]    # chosen rows, sliced columns
m[:, [0, 2]]      # all rows, chosen columns
```

In [ ]:
import numpy as np

grades = np.array([[78, 85, 90],
                   [62, 71, 88],
                   [90, 93, 79]])

print("students 0 and 2, subjects 1-2:")
print(grades[[0, 2], 1:3])
print()
print("every student, subjects 0 and 2:")
print(grades[:, [0, 2]])

## 9. Setting Values with Masks

Masks are not only for reading — assign through them to update many elements in one statement. This replaces whole if/else loops and powers everyday data cleaning: clipping outliers, zeroing invalid readings, marking failures.

**Syntax:**
```python
arr[arr > limit] = cap        # cap every value above limit
arr[mask] = new_value         # set all True positions
```

In [ ]:
import numpy as np

sensor = np.array([12, 98, 15, 240, 18, 5, 190])   # readings with spikes

clean = sensor.copy()      # copy so the raw data survives for comparison
clean[clean > 100] = 100   # clip every spike down to 100
print("raw  :", sensor)
print("fixed:", clean)

board = np.zeros((3, 3), dtype=int)
board[board == 0] = 7      # every zero becomes 7
print(board)

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Combining conditions with `and` / `or` / `not` | `ValueError: The truth value of an array is ambiguous` | Use `&`, `|`, `~` with parentheses around each comparison |
| Forgetting slices are views | Editing a slice silently rewrites the original array | Call `.copy()` when you need independence |
| Writing `m[i][j]` | Two lookups and an intermediate row object; breaks some assignment patterns | Use the tuple form `m[i, j]` |
| Assuming `arr[arr > 5]` gives a view | Boolean masking ALWAYS returns a copy — edits vanish | Assign back through the mask: `arr[arr > 5] = 0` |
| `scores[10]` on a 6-element array | `IndexError: index 10 is out of bounds` | Check `arr.shape` / `len(arr)` before hard-coding positions |

## 💡 Best Practices & Pro Tips

- Prefer the tuple form `arr[row, col]` everywhere — it is faster and it is what all documentation uses.
- Name your masks (`passed = scores >= 60`) instead of repeating raw comparisons; future-you will thank present-you.
- Burn in the rule: **basic slicing → view, boolean/fancy indexing → copy**. It explains 90% of "why did my array change?" mysteries.
- Keep masks small and composable: build `(x >= low) & (x <= high)` step by step when debugging.
- **AI-engineering relevance:** mask filtering is dataset surgery — `X[y == 1]` grabs all positive-class rows, and attention mechanisms / top-k retrieval are built on fancy indexing.

## 📌 Summary

| Operation | What it does | Example |
|---|---|---|
| `arr[i]` / `arr[-1]` | Single element (negatives from end) | `scores[0]`, `scores[-1]` |
| `arr[a:b:c]` | Slice — a **view** of the original | `scores[1:4]` |
| `.copy()` | Independent copy of a slice | `scores[1:4].copy()` |
| `m[r, c]` | 2-D element, tuple form | `grid[1, 2]` |
| `m[r, :]` / `m[:, c]` | Entire row / column | `grades[:, 0]` |
| `m[a:b, c:d]` | Rectangular sub-block | `grades[0:2, 0:2]` |
| `arr[cond]` | Boolean mask filter (a copy) | `temps[temps > 32]` |
| `&` `\|` `~` | Combine / negate masks | `x[(x>0) & (x<10)]` |
| `arr[[i, j, k]]` | Fancy indexing by positions | `prices[[3, 0, 3]]` |
| `arr[cond] = v` | Masked assignment | `sensor[sensor > 100] = 100` |

Key takeaways:
- Slices are windows into existing memory; masks and fancy indices assemble new arrays.
- One bracket tuple beats chained brackets on 2-D arrays.
- Masks turn questions ("which values pass?") into selections — the backbone of data filtering.
- Anything you can read with an index expression, you can also assign to.

## 🔗 Next Lesson

- Continue to **[03_DataTypes_Copy_View](../03_DataTypes_Copy_View/notes.ipynb)** — dtypes, silent integer overflow, and the full story behind copies and views.